In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv(r"D:\Khushi\Cleansing_Data\us\texas\tarrant\transaction_log\Raw_Data\us_tx_48439_data_trxlog_main_bronze_20260218.csv")
df

In [ ]:
df.info()

In [ ]:
df[df['input_book']==0.0]

In [ ]:
df.info()

In [ ]:
# Define fake null values
fake_nulls = ['', 'null', 'none', 'nan', "None", 'na', 'n/a', 'n.a.', 'NaN', 'NAN', 'NULL', 'nil', 'Nan', 'Null','NONE']

# ✅ Now check and replace in all columns
for col in df.columns:
    mask = df[col].astype(str).str.lower().isin([str(x).lower() for x in fake_nulls])
    problem_rows = df.loc[mask, [col]]

    if not problem_rows.empty:
        print(f"Rows where '{col}' has fake nulls:")
        print(problem_rows)

        df[col] = df[col].replace(fake_nulls, np.nan)
print("\nNaN count per column:")

print(df.isna().sum())
print("\nCleaned DataFrame:")
print(df)

In [ ]:
import re
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].apply(
        lambda x: re.sub(r'\s+', ' ', x).strip() if isinstance(x, str) else x
    )

In [ ]:
for col in df.columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=True))

In [ ]:
df[df['instrument']=='D209261910']

In [ ]:
df[df['bk_pg']=='10550_1724']

In [ ]:
df.columns

In [ ]:
df = df.rename(columns={'bk_pg':'book_page'})

In [ ]:
df[df['grantee'].str.len()==1]

In [ ]:
df[df['grantor'].str.len()==1]

In [ ]:
df[df['grantee'].str.len()==2]

## Replacing the 'G','C'and 'UP' with null from column 'grantee'

In [ ]:
df['grantee'] = df['grantee'].replace(['G','C','UP'],np.nan)
df['grantee']

In [ ]:
df[df['grantee'].str.len()==2]

In [ ]:
df[df['grantor'].str.len()==2]

## Replacing the 'G','.','E','M','BA' and 'UD' with null from column 'grantor'

In [ ]:
df['grantor'] = df['grantor'].replace(['G','.','M','E','BA','UD'],np.nan)
df['grantor']

In [ ]:
df[df['grantor'].str.len()==1]

In [ ]:
df[df['grantor'].str.len()==2]

In [ ]:
df[df['grantee'].str.len()==1]

In [ ]:
df[df['input_instrument'].isna()]

In [ ]:
df[df['book_page'].isna()]

In [ ]:
df[df['book/liber/page']=='--/--/--']

## Replacing the '--/--/--' with null from column 'book/liber/page' 

In [ ]:
df['book/liber/page'] = df['book/liber/page'].replace('--/--/--',np.nan)
df['book/liber/page']

In [ ]:
df[df['book/liber/page']=='--/--/--']

In [ ]:
print(df['book/liber/page'].value_counts().to_string())

In [ ]:

df["book/liber/page"] = df["book/liber/page"].apply(lambda x: str(x).replace("/--/--", "") if pd.notna(x) else x)

df['book/liber/page']

In [ ]:
print(df['book/liber/page'].value_counts().to_string())

In [ ]:
df.info()

## Change date formtting from column 'recorded_date'

In [ ]:
df['recorded_date'] = pd.to_datetime(df['recorded_date'],errors='raise')
df['recorded_date']

In [ ]:
df.info()

## Change type into intger from column 'number_of_pages'

In [ ]:
df['number_of_pages'] = pd.to_numeric(df['number_of_pages'], errors='raise').astype('Int64')
df['number_of_pages']

In [ ]:
df.info()

In [ ]:
checks = {
    "Numeric 0": 0,
    "String '0'": '0',
    "Hyphen '-'": '-',
    "Dot '.'": '.',
    "Comma ','": ',',
    "Backtick '`'": '`'
}

for label, val in checks.items():
    print(f"\n=== {label} ===")
    result = (df == val).sum().loc[lambda x: x > 0]
    print(result if not result.empty else "No matches found")

In [ ]:
df['number_of_pages'] = df['number_of_pages'].replace(0,np.nan)
df['number_of_pages']

In [ ]:
df.columns

In [ ]:
cole = ['input_instrument','input_book', 'input_page', 'instrument', 'book_page', 'recorded_date', 'number_of_pages',
       'book/liber/page', 'grantor', 'grantee', 
       'deed_document_present', 
       'image_save','page_save']
df = df.reindex(columns=cole)

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df

In [ ]:
df.columns

In [ ]:
col = df.columns.difference(['input_instrument','input_book','input_page','deed_document_present', 'image_save', 'page_save'])
mask = df[col].isna().all(axis=1)
df[mask]

## Remove the main not found record

In [ ]:
removed_df = df.loc[mask]
df = df.loc[~mask].reset_index(drop=True)

In [ ]:
col = df.columns.difference(['input_instrument','input_book','input_page','deed_document_present', 'image_save', 'page_save'])
mask = df[col].isna().all(axis=1)
df[mask]

In [ ]:
df.info()

In [ ]:
df.columns

## Drop duplicate column ''

In [ ]:
colsss = ['input_instrument', 'input_book', 'input_page',]
df.drop(columns=colsss , inplace=True)

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated()]

## Drop duplicated 24 records

In [ ]:
df = df.drop_duplicates(keep='first')

In [ ]:
df.duplicated().sum()

In [ ]:
df

In [ ]:
df.info()

## Placing the file

In [ ]:
df.to_csv(r'D:\Khushi\Cleansing_Data\us\texas\tarrant\transaction_log\Cleaned_Data\us_tx_48439_data_trxlog_main_silver_20260218.csv',index=False)

In [ ]:
df['grantee'].value_counts()

In [ ]:
df['grantor'].value_counts()

In [ ]:
df.columns

In [ ]:
df[df['instrument']=='D220239232']

## Genrating the transaction party csv

In [ ]:
selected=df[['instrument', 'book_page','book/liber/page', 'grantor', 'grantee',]]
selected

In [ ]:
nan_in_grantor_grantee = df[df[['grantor','grantee']].isna().any(axis=1)]
nan_in_grantor_grantee

In [ ]:
df1= df

## Removing the nan record from column 'grantor' and 'grantee'

In [ ]:
selected = df1[['instrument', 'book_page','book/liber/page', 'grantor', 'grantee',]].dropna(subset=['grantor','grantee'])
selected

In [ ]:
selected['grantor'].isna().sum()

In [ ]:
selected['grantee'].isna().sum()

In [ ]:
def create_role_name(selected):
    rows = []
    for idx, row in selected.iterrows():
        grantors = str(row['grantor']).split('||')
        grantees = str(row['grantee']).split('||')
 
        for g in grantors:
            rows.append({
                "instrument": row["instrument"],
                "book_page": row["book_page"],
                "book/liber/page":row["book/liber/page"],
                "role": "grantor",
                "name": g.strip()
            })
 
        for r in grantees:
            rows.append({
                "instrument": row["instrument"],
                "book_page": row["book_page"],
                "book/liber/page":row["book/liber/page"],
                "role": "grantee",
                "name": r.strip()
            })
    return pd.DataFrame(rows)
final_df = create_role_name(selected)
print(final_df)

In [ ]:
nan_in_grantor1 = final_df[final_df[['name']].isna().any(axis=1)]
nan_in_grantor1

In [ ]:
selected[selected['instrument']=='D165046805']

In [ ]:
final_df[final_df['instrument']=='D165046805']

## placing the transaction_party csv

In [ ]:
final_df.to_csv(r"D:\Khushi\Cleansing_Data\us\texas\tarrant\transaction_log\Cleaned_Data\TransactionParty.csv", index=False)
final_df

In [ ]:
final_df['instrument'].nunique()

In [ ]:
final_df['book_page'].nunique()